=======================================================================================================================
#### Gathering Data with Web Scraping

Scraping from:
* Amazon
* Newegg
* BestBuy

Collecting:
- Title
- Rating
- Rating Count
- Price
- URL
- Site
  
=======================================================================================================================

In [47]:
from bs4 import BeautifulSoup as BS

from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
import undetected_chromedriver as UC
import time
import random
from tqdm import tqdm
import json
from selenium.webdriver.support import expected_conditions as EC

import random
from concurrent.futures import ThreadPoolExecutor, as_completed, ProcessPoolExecutor
from threading import Lock

import csv

In [3]:
#Implement Roating Proxies and User Agents

def newegg_scrape_page(drivers, start_url):
    try:
        drivers.get(str(start_url))

        WebDriverWait(drivers, 5).until(
            lambda d: d.execute_script('return document.readyState') == 'complete'
        )

        final_products = []
        soup = BS(driver.page_source, 'lxml')
        all_items = soup.find_all('div', class_ = 'item-cell')
        for item in all_items:
            title_tag = item.find('a', class_ = 'item-title')
            title = title_tag.text.strip() if title_tag else "N/A"

            rating_tag = item.find('i', class_ = 'rating rating-4-5')
            rating = rating_tag.get('aria-label') if rating_tag else "N/A"

            rating_count_tag = item.find('span', class_ = 'item-rating-num')
            rating_count = rating_count_tag.text.strip('()') if rating_count_tag else "N/A"

            price_tag = item.find('li', class_ = 'price-current').find('strong')
            price = price_tag.text.strip() if price_tag else "N/A"

            URL_tag = item.find('a', class_ = 'item-title')
            URL = URL_tag.get('href') if URL_tag else "N/A"

            final_products.append({
                'Title': title,
                'Rating': rating,
                'Rating Count': rating_count,
                'Price': price,
                'URL': URL,
                'Site': 'Newegg'
            })
        
        return final_products
    except Exception as e:
        print(e)
        return

In [4]:
def amazon_scrape_page(driver, start_url):
    try:
        driver.get(str(start_url))

        WebDriverWait(driver, 5).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )

        time.sleep(random.uniform(1, 3))

        final_products = []
        soup = BS(driver.page_source, 'lxml')
        all_items = soup.find_all('div', {
            'role': lambda x: x and x.startswith('listitem'),
            'data-asin': True
        })
        for i, item in enumerate(all_items):
            try:
                title_tag = item.find('a', class_ = 's-line-clamp-2').find('h2')
                #print(title_tag)
                title = title_tag.find('span').text.strip() if title_tag else "N/A"
                #print(title)

                rating_tag = item.find('i', class_ = 'a-icon').find('span')
                rating = rating_tag.text.strip('out of 5 stars') if rating_tag else "N/A"
                #print(rating)

                rating_count_tag = item.find('a', class_ = "s-underline-text").get('aria-label')
                rating_count = rating_count_tag.strip('ratings') if rating_count_tag else "N/A"

                price_tag = item.find('span', class_ = 'a-price-whole')
                #print(price_tag)
                price = price_tag.text.strip() if price_tag else "N/A"
                #print(price)

                URL_tag = item.find('a', class_ = 's-line-clamp-2')
                #print(URL_tag)
                URL = URL_tag.get('href') if URL_tag else "N/A"
                #print(URL)

                final_products.append({
                    'Title': title,
                    'Rating': rating,
                    'Rating Count': rating_count,
                    'Price': price,
                    'URL': f"https://www.amazon.com{URL}",
                    'Site': 'Amazon'
                })
                
            except Exception as product_error:  # ← Catch per-product errors
                print(f"Skipping bad product: {product_error}")
                #final_products.append({'Title': 'Parse Error', 'Site': 'Amazon'})
                continue  # Move to next product

        #print(f'Successfully scraped {len(final_products)} products - Amazon!')
        return final_products

    except Exception as e:
        print(e)

In [5]:
def bestbuy_scrape_page(driver, start_url):
    try:
        driver.get(str(start_url))

        WebDriverWait(driver, 5).until(
            lambda d: d.execute_script('return document.readyState') == 'complete'
        )

        last_height = driver.execute_script('return document.body.scrolHeight')

        for i in range(8):
            driver.execute_script(f'window.scrollTo(0, {(i+1) * 500});')
            time.sleep(random.uniform(10, 12))

        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(1)

        driver.execute_script('window.scrollTo(0, 0);')

        soup = BS(driver.page_source, 'lxml')
        final_products = []
        all_items = soup.find_all('li', class_ = 'product-list-item grid-view')
        for i, item in enumerate(all_items):
            print(f'Trial Number: {i+1}/{len(all_items)}')
            link_tag = item.find('a', class_ = 'product-list-item-link')
            if not link_tag:
                print(f"Skipping item {i} - no product link")
                continue
    
            
            title_tag = item.find('a', class_ = 'product-list-item-link').find('h3')
            title = title_tag.get_text() if title_tag else "N/A"

            rating_tag = item.find('div', class_ = 'c-ratings-reviews').find('span', class_ = 'font-weight-medium')
            rating = rating_tag.text if rating_tag else "N/A"

            review_tag = item.find('span', class_='c-ratings-reviews-mini mt-25') or \
                         item.find('span', class_='c-reviews order-2')
            review_count = 'N/A'
            if review_tag:
                review_text = review_tag.text.strip('()')
                review_count = review_text

            price_tag = item.find('div', {'data-testid':'price-block-customer-price'})
            #print(price_tag)
            price = price_tag.find('span').text.strip('$') if price_tag else "N/A"
            #print(price)

            URL_tag = item.find('a', class_ = 'product-list-item-link')
            URL = URL_tag.get('href') if URL_tag else "N/A"
            #print(URL)

            final_products.append({
                'Title': title,
                'Rating': rating,
                'Rating Count': review_count,
                'Price': price,
                'URL': URL,
                'Site': 'BestBuy'
            })

        

        driver.quit()
        print(f"Successfully scraped {len(final_products)} from BestBuy!")
        return final_products
            

    except Exception as e:
        print(e)
        traceback.print_exc()
        return[]

In [7]:
def get_driver(user_agent = None):
    options = UC.ChromeOptions()
    options.add_argument('--headless=new')
    options.add_argument('--window-size=1920x1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument(f'--user-agent={user_agent}')

    driver = UC.Chrome(options = options, version_main = 144)
    return driver

In [36]:
driver_lock = Lock()

def scrape_price_range_newegg(price_range, ua_pool):
    product = []
    with driver_lock:
        driver = get_driver(user_agent = random.choice(ua_pool))
    try:
        for i in range(20):
            result = newegg_scrape_page(driver, f'https://www.newegg.com/p/pl?d=pre+built+gaming+pcs&LeftPriceRange={price_range[0]}+{price_range[1]}&page={i+1}')
            if result is not None:
                product.append(result)
    finally:
        driver.quit()

    return product

In [6]:
ua_pool = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Ubuntu; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36",
]

----------------------------------------------------------------
### <center>CODE BELOW</center>
----------------------------------------------------------------

In [37]:
price_range = [
    (0, 1250),
    (1250, 2500),
    (2500, 5000),
    (5000, 7000)
]

newegg_products = []
with ThreadPoolExecutor(max_workers = 4) as executor:
    futures = [executor.submit(scrape_price_range_newegg, pr, ua_pool) 
               for pr in price_range]

    for future in tqdm(as_completed(futures), total = len(price_range), desc = 'Price Ranges'):
        newegg_products.extend(future.result())

newegg_products = [item for page in newegg_products for item in page]

Price Ranges: 100%|██████████████████████████████████████████████████████████████████████| 4/4 [01:15<00:00, 18.80s/it]


In [38]:
len(newegg_products)

2880

In [40]:
amazon_products = []
driver = get_driver(user_agent = random.choice(ua_pool))
for i in tqdm(range(20), desc = 'Scraping Amazon'):
    time.sleep(random.uniform(1,3))
    result = amazon_scrape_page(driver, f'https://www.amazon.com/s?k=pre+built+pc+gaming&page={i+1}&low-price=0&high-price=2500')

    if result is not None:
        amazon_products.append(result)
driver.quit()

driver = get_driver(user_agent = random.choice(ua_pool))
for i in tqdm(range(20), desc = 'Scraping Amazon'):
    time.sleep(random.uniform(1,3))
    result = amazon_scrape_page(driver, f'https://www.amazon.com/s?k=pre+built+pc+gaming&page={i+1}&low-price=2500&high-price=5000')

    if result is not None:
        amazon_products.append(result)
driver.quit()

driver = get_driver(user_agent = random.choice(ua_pool))
for i in tqdm(range(20), desc = 'Scraping Amazon'):
    time.sleep(random.uniform(1,3))
    result = amazon_scrape_page(driver, f'https://www.amazon.com/s?k=pre+built+pc+gaming&page={i+1}&low-price=5000&high-price=7500')

    if result is not None:
        amazon_products.append(result)
driver.quit()


amazon_products = [product for page in amazon_products for product in page]

Scraping Amazon:  60%|███████████████████████████████████████                          | 12/20 [01:26<00:54,  6.83s/it]

Skipping bad product: 'NoneType' object has no attribute 'find'


Scraping Amazon:  90%|██████████████████████████████████████████████████████████▌      | 18/20 [02:03<00:11,  5.96s/it]

Skipping bad product: 'NoneType' object has no attribute 'find'


Scraping Amazon: 100%|█████████████████████████████████████████████████████████████████| 20/20 [02:15<00:00,  6.76s/it]

...


Scraping Amazon:  25%|████████████████▌                                                 | 5/20 [00:32<01:36,  6.42s/it]

Skipping bad product: 'NoneType' object has no attribute 'find'


Scraping Amazon:  30%|███████████████████▊                                              | 6/20 [00:37<01:22,  5.88s/it]

Skipping bad product: 'NoneType' object has no attribute 'find'
Skipping bad product: 'NoneType' object has no attribute 'find'
Skipping bad product: 'NoneType' object has no attribute 'find'


Scraping Amazon: 100%|█████████████████████████████████████████████████████████████████| 20/20 [01:50<00:00,  5.55s/it]


In [41]:
len(amazon_products)

786

In [43]:
driver = get_driver(user_agent = random.choice(ua_pool))

bestbuy_products = bestbuy_scrape_page(driver, 'https://www.bestbuy.com/site/searchpage.jsp?cp=1&id=pcat17071&st=pre+built+gaming+pcs')

Trial Number: 1/18
Trial Number: 2/18
Trial Number: 3/18
Trial Number: 4/18
Trial Number: 5/18
Skipping item 4 - no product link
Trial Number: 6/18
Skipping item 5 - no product link
Trial Number: 7/18
Skipping item 6 - no product link
Trial Number: 8/18
Skipping item 7 - no product link
Trial Number: 9/18
Skipping item 8 - no product link
Trial Number: 10/18
Skipping item 9 - no product link
Trial Number: 11/18
Skipping item 10 - no product link
Trial Number: 12/18
Skipping item 11 - no product link
Trial Number: 13/18
Skipping item 12 - no product link
Trial Number: 14/18
Skipping item 13 - no product link
Trial Number: 15/18
Skipping item 14 - no product link
Trial Number: 16/18
Skipping item 15 - no product link
Trial Number: 17/18
Skipping item 16 - no product link
Trial Number: 18/18
Skipping item 17 - no product link
Successfully scraped 4 from BestBuy!


In [44]:
bestbuy_products

[{'Title': 'Skytech Gaming - Skytech ARCH4 - AMD Ryzen 7 5700, 5050 8GB, 16GB 3200 MHz DDR4 SDRAM, 650W GOLD PSU - White',
  'Rating': '4.7',
  'Rating Count': '244 reviews from https://skytechgaming.com/',
  'Price': '999.99',
  'URL': 'https://www.bestbuy.com/product/skytech-gaming-skytech-arch4-amd-ryzen-7-5700-5050-8gb-16gb-3200-mhz-ddr4-sdram-650w-gold-psu-white/J3GWRTR68X',
  'Site': 'BestBuy'},
 {'Title': 'Velztorm - Pilum Pre-Built Gaming Desktop PC (AMD Ryzen 7 5700X, GeForce RTX 5060, 32GB Memory, 1TB SSD, Liquid Cool, Windows 11) - Black',
  'Rating': '5.0',
  'Rating Count': '1 review',
  'Price': '1,049.99',
  'URL': 'https://www.bestbuy.com/product/velztorm-pilum-pre-built-gaming-desktop-pc-amd-ryzen-7-5700x-geforce-rtx-5060-32gb-memory-1tb-ssd-liquid-cool-windows-11-black/JJGHT42R6G',
  'Site': 'BestBuy'},
 {'Title': 'Skytech Gaming - KING95 Gaming Desktop PC - AMD Ryzen 7 9700X - 16GB Memory - NVIDIA GeForce RTX 5060 Ti 16GB - 1TB NVMe SSD - White',
  'Rating': '4.7',
 

In [45]:
final_products = newegg_products + amazon_products + bestbuy_products

In [46]:
len(final_products)

3670

In [50]:
with open('unclean_products.csv', 'w', newline = '', encoding = 'utf-8') as file:
    writer = csv.DictWriter(file, fieldnames = final_products[0].keys())
    writer.writeheader()
    writer.writerows(final_products)